[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [Peewee, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/peewee-deep-dive.html)

# Migrations &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

The first cell is the notebook's Setup with two commands added, so that the project already has its
first migration written and applied. Run it first, then the tasks in order, since each one leaves
the project where the next one expects it.


In [1]:
import re
import subprocess
import sys
import tempfile
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path

try:
    if version("peewee") != "4.5.1":                                # Colab has 4.4.0, whose wording differs
        raise PackageNotFoundError
except PackageNotFoundError:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--root-user-action=ignore",
                    "peewee==4.5.1"], check=True)

import peewee
from peewee import SqliteDatabase
from playhouse.reflection import generate_models, print_model

WORK = Path(tempfile.mkdtemp(prefix="catalog-"))                    # the project, for this session only
PWMIGRATE = f"{sys.executable} -m playhouse.migrations"             # the same tool the pwmigrate script runs

FIRST = """from peewee import CharField, ForeignKeyField, IntegerField, Model, SqliteDatabase

db = SqliteDatabase("app.db", pragmas={"foreign_keys": 1})


class Author(Model):
    name = CharField(max_length=60, unique=True)

    class Meta:
        database = db


class Note(Model):
    title = CharField(max_length=80)
    author = ForeignKeyField(Author, backref="notes")

    class Meta:
        database = db
"""


def run(command):
    """Run a pwmigrate command in the project and print what it printed, without the clock."""
    done = subprocess.run(f"{PWMIGRATE} {command}", shell=True, cwd=WORK,
                          capture_output=True, text=True)
    printed = (done.stdout + done.stderr).strip()
    print(re.sub(r"\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}", "<applied at>", printed) or "(no output)")
    return done.returncode


def show(name):
    """Print a file from the project, with the generator's date taken out."""
    text = (WORK / name).read_text()
    print(re.sub(r"on \d{4}-\d{2}-\d{2} \d{2}:\d{2}", "on <when>", text).rstrip())


def write_models(text):
    """Replace models.py, which is what a schema change looks like before any migration exists."""
    (WORK / "models.py").write_text(text)


write_models(FIRST)
(WORK / "app.db").touch()                                           # pwmigrate needs the file to exist

print("peewee", peewee.__version__)
print("the project holds:", sorted(path.name for path in WORK.iterdir()))


run("app.db initial models")
run("app.db up")
print("ready:", sorted(path.name for path in (WORK / "migrations").glob("*.py")))


peewee 4.5.1
the project holds: ['app.db', 'models.py']
migrations/0001_initial.py
applied: 0001_initial
ready: ['0001_initial.py']


**1.** What has run.


In [2]:
run("app.db status")
print()
print("the migrations on disk:", sorted(path.name for path in (WORK / "migrations").glob("*.py")))


[x] 0001_initial  <applied at>

the migrations on disk: ['0001_initial.py']


One migration, written by `initial` and applied by `up`. The `[x]` and the timestamp come from the
history table in `app.db` rather than from the folder, which is why the two lists can disagree and
why `status` is the one to trust about what has happened.


**2.** A field, a difference, and a file.


In [3]:
write_models((WORK / "models.py").read_text().replace(
    '    author = ForeignKeyField(Author, backref="notes")\n',
    '    author = ForeignKeyField(Author, backref="notes")\n'
    "    pinned = IntegerField(null=True)\n"))

print("diff:")
run("app.db diff models")
print()
run("app.db generate add_pinned models")
run("app.db status")


diff:
add column note.pinned

migrations/0002_add_pinned.py
[x] 0001_initial  <applied at>
[ ] 0002_add_pinned


1

The file exists and the database has not changed: `generate` writes, and only `up` applies. The `[ ]`
is the proof.


**3.** The two halves of the file.


In [4]:
show("migrations/0002_add_pinned.py")


# Generated from a schema diff on <when>.
from peewee import *

def up(migrator, db):
    migrator.migrate(migrator.add_column('note', 'pinned', IntegerField(null=True)))


def down(migrator, db):
    migrator.migrate(migrator.drop_column('note', 'pinned'))


One line in each function, and they are a pair: `add_column('note', 'pinned', ...)` in `up` is undone
by `drop_column('note', 'pinned')` in `down`. Where `up` has several lines, `down` has them in the
reverse order, so that anything depending on something else is removed before the thing it depends
on.


**4.** There and back.


In [5]:
run("app.db up")
print()
print("diff after applying:")
run("app.db diff models")

print()
run("app.db down")
print()
print("diff after reverting:")
run("app.db diff models")


applied: 0002_add_pinned

diff after applying:
schema matches models.

reverted: 0002_add_pinned

diff after reverting:
add column note.pinned


0

`schema matches models.` and then the difference again. The models file never changed: what changed
is the database, twice, and the difference is the distance between them.


**5.** The exit code.


In [6]:
print("with 0002 pending  -> exit", run("app.db status"))
print()
run("app.db up")
print("with nothing pending -> exit", run("app.db status"))


[x] 0001_initial  <applied at>
[ ] 0002_add_pinned
with 0002 pending  -> exit 1

applied: 0002_add_pinned
[x] 0001_initial  <applied at>
[x] 0002_add_pinned  <applied at>
with nothing pending -> exit 0


One when a migration exists and has not run, zero when they all have. That is the whole check: a
build step that runs `pwmigrate app.db status` fails on a database that has not caught up.


**6.** The schema, read back out.


In [7]:
found = generate_models(SqliteDatabase(str(WORK / "app.db")))

print("tables:", sorted(found))
print("note:  ", [field.name for field in found["note"]._meta.sorted_fields])
print("author:", [field.name for field in found["author"]._meta.sorted_fields])


tables: ['author', 'note', 'schema_migration']
note:   ['id', 'title', 'author', 'pinned']
author: ['id', 'name']


No `models.py` was read to produce that. `schema_migration` in the table list is the history table,
which lives in your database like any other, and is the thing `status` was reading all along.


---

&#8592; **Back to:** [Migrations](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/peewee-deep-dive/10-migrations.ipynb)  &nbsp;&middot;&nbsp;  [Peewee, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/peewee-deep-dive.html)
